In [ ]:
library(ggplot2)
library(furrr)
library(nortest)

In [ ]:
N = 300 # 50

contamination_proportion <- floor(N * 20/100)

x_vals = seq(0, N)

set.seed(0)
small_noise <- rnorm(n=N+1, mean=0, sd=1)
#big_noise   = rnorm(n=contamination_proportion, mean=25, sd=50)
#big_noise   = runif(n=contamination_proportion, min = -20, max=20)
#big_noise <- 10*rcauchy(n=contamination_proportion,scale = 1, location = 0)
#big_noise <- rlnorm(n=contamination_proportion, meanlog=2.5, sdlog = 0.3)
#big_noise <- rep(0,times = contamination_proportion)

y_vals = 1 * x_vals + small_noise

#random_sample <- sample(1:length(x_vals), size=contamination_proportion)

#y_vals[random_sample] = y_vals[random_sample] + big_noise



In [ ]:
slope <- function(x,y,i,j) {
    if (x[i] == x[j]) {

        assert(0 == 1)

    } else {

    }
    return ((y[j] - y[i]) / (x[j] - x[i]))
}

repeated_median_estimator <- function(x, y) {

    inner_slopes <- numeric(length(x))

    for (i in 1:length(x)) {
        # using c() for convenience
        # we know the number of elements in advance, so numeric(...) works also,
        # but the i==j case leaves the corresponding element as 0 - which has an effect on the median() at the end.
        # Alternatively we could track the median through the iterations and insert into the list at the right place - is it better?
        temp_slopes <- c() 

        for (j in 1:length(y)) {

            if (i == j) {
                next
            } else {
                temp_slopes = c(temp_slopes, slope(x,y,i,j))
            }

        }
        inner_slopes[i] <- median(temp_slopes)
    }

    B_hat = median(inner_slopes)
    A_hat = median(y - B_hat*x) # see method parameter in https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.theilslopes.html#scipy.stats.theilslopes

    return(list(B=B_hat, A=A_hat))

}

theil_sen_estimator <- function(x, y) {
    inner_slopes <- c()

    for (i in 1:(length(x)-1)) {
        for (j in (i+1):length(y)) {
            if (i >= j) {
                next
            } else {
                inner_slopes <- c(inner_slopes, slope(x,y,i,j))
            }
        }
    }

    B_hat = median(inner_slopes)
    A_hat = median(y - B_hat*x) # see method parameter in https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.theilslopes.html#scipy.stats.theilslopes

    return(list(B=B_hat, A=A_hat))

}

theil_estimator <- theil_sen_estimator(x_vals, y_vals)

#median_estimator <- repeated_median_estimator(x_vals, y_vals)



In [ ]:
huber_psi <- function(x, k = 1.345) {
    # constant for 95% normal efficiency of the estimator

    return (ifelse(abs(x) <= k, x, k * sign(x)))
}

bisquare_psi <- function(x, k = 1.345) {
    ifelse(abs(x) <= k, (6 / k^2) * x * (1 - (x/k)^2)^2, 0)
}


bisquare_weight <- function(x, k = 1.345) {
    ifelse(abs(x) <= k, (1 - (x/k)^2)^2, 0)
}

huber_weight <- function(x, k=1.345) {
    ifelse(abs(x) <= k, 1, k / abs(x))
}

madn <- function(x) {
    c = 0.675 # Phi^-1(0.75) normal dist

    med = median(x, na.rm = TRUE)
    return(1/c * median(abs(x - med), na.rm = TRUE))
}

IRLS_solve <- function(x, y, weight_fn, max_iter = 1000, eps = 1e-6) {

    stopifnot(length(x) == length(y))

    X <- cbind(1, x)

    # initial estimate should be L1 for faster convergence, and for redescending estimators not using L1 may mean convergence to a bad solution
    # for simplicity we can use OLS here so long as our psi is convex

    fit <- lm.fit(X, y)

    sigma_initial <- madn(residuals(fit))

    if (sigma_initial == 0) {
        med <- median(residuals(fit))
        if (med == 0) {
            print("ERROR")
            stopifnot(1==0)
        }
    }

    for (k in 1:max_iter) {
        resid <- residuals(fit, type = "response")
        candidate_weights <- weight_fn(resid / sigma_initial)

        

        candidate_fit <- lm.wfit(X, y, candidate_weights) # using lm naively is much slower here

        if (max(abs(residuals(fit) - residuals(candidate_fit))) < eps) {
            break
        } else {
            fit <- candidate_fit
        }


    }
    return(fit)

}


In [ ]:
huber_m_fit <- coef(IRLS_solve(x_vals,y_vals, huber_weight))

options(repr.plot.width = 20, repr.plot.height = 10, repr.plot.res = 100)
ggplot(data = data.frame(x_vals, y_vals), aes(x=x_vals, y=y_vals)) +
    geom_point() +
    geom_abline(aes(intercept=huber_m_fit[1], slope=huber_m_fit[2]), color="red", linewidth=1) +
    geom_abline(aes(intercept=theil_estimator$A, slope=theil_estimator$B), color="purple", linewidth=1) +
    geom_abline(aes(intercept=0, slope=1), color="green", size=1) +
    geom_smooth(method='lm', formula= y~x, color="blue4")

In [ ]:
simulation <- function(i) {
    x_vals = seq(0, N)
    small_noise <- rnorm(n=N+1, mean=0, sd=1)
    y_vals = 1 * x_vals + small_noise

    # No contamination for this model

    huber_beta <- coef(IRLS_solve(x_vals,y_vals, huber_weight))[2]

    return(huber_beta)
}
N_sim <- 10000
plan(multisession, workers = 8)


results <- future_map_dbl(1:N_sim, simulation, .options = furrr_options(seed = 0))

In [ ]:
mean((results - 1)^2)
mean(results)
var(results)
hist(results)

ad.test(results)
qqnorm(results)


In [ ]:
source("functions.R")

In [ ]:
IRWLS_fit_simple(x_vals, y_vals, initial_est = "S", weight_fn = function(x) bisquare_weight(x, k=4.68))

In [ ]:
N <- 300

simulation <- function(i) {
    x_vals = seq(0, N)
    small_noise <- rnorm(n=N+1, mean=0, sd=1)
    y_vals = 1 * x_vals + small_noise

    contamination_proportion <- floor(N * 45/100)


    #big_noise   = rnorm(n=contamination_proportion, mean=20, sd=100)
    #big_noise <- runif(n=contamination_proportion, min= -40, max=10)

    #random_sample <- sample(1:length(x_vals), size=contamination_proportion)

    #y_vals[random_sample] = y_vals[random_sample] + big_noise


    bi_fit <- IRWLS_fit_simple(x_vals,y_vals, initial_est = "S", weight_fn = function(x) bisquare_weight(x, k=4.68))
    ols_fit <- lm(y_vals ~ x_vals)

    return(list(
        bisquare_slope = coef(bi_fit)[2],
        ols_slope      = coef(ols_fit)[2]
    ))
}
N_sim <- 10000
plan(multisession, workers = 8)


results <- future_map(1:N_sim, simulation, .options = furrr_options(seed = 0))

In [ ]:
bisquare_slopes <- sapply(results, `[[`, "bisquare_slope")
ols_slopes      <- sapply(results, `[[`, "ols_slope")

par(bg = "white")
hist(bisquare_slopes)
mean((bisquare_slopes - 1)^2)
var(bisquare_slopes)
ad.test(bisquare_slopes)


In [ ]:
par(bg = "white")

hist(ols_slopes)
mean((ols_slopes - 1)^2)
var(ols_slopes)
ad.test(ols_slopes)

In [ ]:
var(ols_slopes) / var(bisquare_slopes) # roughly 95% is a good sign

In [ ]:
source("simulations.R")

In [ ]:
normal_efficiency_sim(N_pts = 100)